In [1]:
import geopandas as gpd
import numpy as np
import osmnx as ox

import focalpy

In [2]:
agglom_extent_filepath = "../data/raw/zurich-extent.gpkg"
sites_gdf_filepath = "../data/processed/stations.gpkg"
lake_nominatim_query = "Zürichsee"

buildings_gdf_filepath = "../data/processed/buildings.gpkg"
tree_canopy_filepath = "../data/processed/tree-canopy.tif"
dem_filepath = "../data/processed/dem.tif"
buffer_dists = [100, 250, 500]
# the TPI is defined by its neighbourhood, so its radius sets the attribute itself and
# both scales carry distinct information
topo_buffer_dists = [250, 500]
# slope and aspect are computed per DEM cell and then averaged, so the radius only sets
# the aggregation extent. both scales correlate at ~0.95 but are kept, like the other
# multi-scale predictors, and left to the VIF-based feature selection to handle any
# redundancy
topo_attr_buffer_dists = [250, 500]

# number of sites processed per batch; reduce if running out of memory
chunk_size = 500

dst_filepath = "../data/processed/stations-features.csv"

In [3]:
# Parameters
buffer_dists = [100, 250, 500]
agglom_extent_filepath = "data/processed/zurich-lcs-extent.gpkg"
chunk_size = 500
sites_gdf_filepath = "data/processed/stations.gpkg"
buildings_gdf_filepath = "data/processed/buildings.gpkg"
tree_canopy_filepath = "data/processed/tree-canopy.tif"
dem_filepath = "data/processed/dem.tif"
dst_filepath = "data/processed/stations-features.csv"


In [4]:
region = gpd.read_file(agglom_extent_filepath).buffer(buffer_dists[-1]).iloc[:1]
lake_extent_gdf = ox.geocode_to_gdf(lake_nominatim_query).to_crs(region.crs)[
    ["geometry"]
]
buildings_gdf = gpd.read_file(buildings_gdf_filepath)
# add a "volume" column
buildings_gdf["volume"] = buildings_gdf.geometry.area * buildings_gdf["height"]

In [5]:
# `site_gser` should already be in EPSG:2056
site_gser = gpd.read_file(sites_gdf_filepath)
sites_index_name = site_gser.columns.drop("geometry")[0]
site_gser = site_gser.set_index(sites_index_name)["geometry"]

In [6]:
def northness(aspect_arr):
    """Compute the mean cosine of an aspect array, i.e., a north-south index in [-1, 1].

    ACHTUNG: aspect is a circular variable, so its arithmetic mean is not a direction:
    cells on either side of north cancel wrongly and the average collapses towards 180
    degrees (south). Averaging the cosine is the standard circular summary.
    """
    return np.mean(np.cos(np.radians(aspect_arr)))


fa = focalpy.FocalAnalysis(
    [
        buildings_gdf,
        tree_canopy_filepath,
        dem_filepath,
        lake_extent_gdf,
        dem_filepath,
        dem_filepath,
    ],
    site_gser,
    [buffer_dists, buffer_dists, [0], [0], topo_attr_buffer_dists, topo_buffer_dists],
    [
        "compute_vector_features",
        "compute_raster_features",
        "compute_elevation_at_site",
        "compute_vector_distance",
        "compute_terrain_attributes",
        # ACHTUNG: the TPI must NOT be computed via `compute_terrain_attributes`. There,
        # `xdem` defines the attribute over a `window_size` cell window (3 by default,
        # i.e., 6 m for our 2 m DEM) and the buffer only sets the extent over which the
        # per-cell values are then averaged, which nearly cancels them out.
        # `compute_topographic_position_index` instead uses the buffer as the
        # neighbourhood that defines the attribute, which is the intended scale
        "compute_topographic_position_index",
    ],
    feature_col_prefixes=["building", "tree", "", "lake", "", ""],
    feature_methods_args={
        "compute_vector_features": [{"volume": "sum"}],
        "compute_terrain_attributes": [["aspect", "slope"]],
    },
    feature_methods_kwargs={
        "compute_raster_features": {"stats": "sum"},
        # ACHTUNG: `stats` applies to every terrain attribute of the call, so this also
        # yields a mean of the circular aspect and a northness of the slope, neither of
        # which is meaningful - both are dropped below
        "compute_terrain_attributes": {"stats": [northness, "mean"]},
    },
    chunk_size=chunk_size,
)

  0%|          | 0/122 [00:00<?, ?it/s]

  0%|          | 0/122 [00:00<?, ?it/s]

In [7]:
# keep only the meaningful attribute-statistic combinations, i.e., the northness of the
# aspect and the mean of the slope. `features_df` is assigned first so that the features
# are computed once
features_df = fa.features_df
features_df = features_df.drop(
    columns=[
        column
        for column in features_df.columns
        if column.startswith("aspect_mean") or column.startswith("slope_northness")
    ]
)
features_df.head()

,building_volume_sum_100,building_volume_sum_250,building_volume_sum_500,tree_sum_100,tree_sum_250,tree_sum_500,elevation_0,lake_distance_0,aspect_northness_250,aspect_northness_500,slope_mean_250,slope_mean_500,topographic_position_index_250,topographic_position_index_500
station_id,,,,,,,,,,,,,,
REH,53579.327844,1.289910e+05,3.275898e+05,690.0,5862.0,29177.0,443.399414,7037.797441,-0.376704,-0.123690,2.149442,3.195710,2.072093,2.629938
SMA,19917.761633,1.838988e+05,6.363434e+05,2390.0,26018.0,122417.0,604.679749,2350.393579,-0.692474,-0.578689,10.031407,9.066966,0.398092,3.504805
Zürich-Kaserne,263396.449315,2.060322e+06,5.209396e+06,1295.0,5673.0,26116.0,406.533997,1495.165182,0.103400,0.096553,2.356884,3.759559,-2.372075,-2.137431
Dübendorf-Empa,113076.299845,6.894265e+05,1.611062e+06,1483.0,9898.0,57244.0,430.772980,6439.998871,-0.048623,-0.012447,5.066118,5.500230,-0.836161,-2.406453
A10,5577.818906,3.334855e+04,3.322713e+05,4286.0,28504.0,118321.0,454.247162,5213.639287,0.382338,0.438507,7.501516,7.932671,-4.185880,-7.650421


In [8]:
features_df.to_csv(dst_filepath)